# npm Hallucination Detector — Ensemble v2 (Colab)

## v1 대비 개선사항
| | v1 | v2 |
|--|--|--|
| 메타 피처 | 10개 | **12개** (Levenshtein 슬롭스쿼팅 특화) |
| CodeBERT 입력 | 패키지명 | **description + 컨텍스트** |
| Benign 데이터 | 유명 패키지만 | **Hard Negative 포함** |
| 추가 피처 | 없음 | **readme_length, has_repo, keywords_count, days_since_created** |

**런타임 → T4 GPU 선택 후 실행**

## 0. 환경 설정

In [ ]:
!git clone https://github.com/haneul-dev/npm-hallucination-detector.git
%cd npm-hallucination-detector
!pip install -q transformers xgboost shap loguru imbalanced-learn

In [ ]:
import sys, os, re, random, time, json, urllib.request, csv
import numpy as np
import pandas as pd
import torch
sys.path.insert(0, '.')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {DEVICE}')

## 1. 데이터 수집 함수 (컨텍스트 피처 포함)

In [ ]:
def fetch_npm_full(name, label=0):
    """npm 레지스트리에서 메타데이터 + 컨텍스트 피처 전체 수집"""
    try:
        req = urllib.request.Request(
            f'https://registry.npmjs.org/{name}',
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req, timeout=6) as r:
            data = json.loads(r.read())

        latest = data.get('dist-tags', {}).get('latest', '')
        ver    = data.get('versions', {}).get(latest, {})
        author = data.get('author', {})

        # 다운로드 수
        try:
            dl_req = urllib.request.Request(
                f'https://api.npmjs.org/downloads/point/last-month/{name}',
                headers={'User-Agent': 'Mozilla/5.0'}
            )
            with urllib.request.urlopen(dl_req, timeout=4) as r2:
                downloads = json.loads(r2.read()).get('downloads', 0)
        except:
            downloads = 0

        # 생성일로부터 경과 일수
        from datetime import datetime, timezone
        created_str = data.get('time', {}).get('created', '')
        try:
            created_dt = datetime.fromisoformat(created_str.replace('Z', '+00:00'))
            days_old   = (datetime.now(timezone.utc) - created_dt).days
        except:
            days_old = 0

        readme = data.get('readme', '') or ''

        return {
            'name':               name,
            'label':              label,
            'exists':             True,
            'author':             author.get('name','') if isinstance(author,dict) else str(author or ''),
            'maintainers_count':  len(data.get('maintainers', [])),
            'dependencies_count': len(ver.get('dependencies', {})),
            'has_install_script': int('install' in ver.get('scripts', {})),
            'downloads':          downloads,
            # 컨텍스트 피처
            'description':        data.get('description', '') or '',
            'has_repository':     int(bool(data.get('repository') or ver.get('repository'))),
            'keywords_count':     len(data.get('keywords', []) or []),
            'readme_length':      len(readme),
            'days_since_created': days_old,
        }
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return {'name': name, 'label': label, 'exists': False,
                    'author': '', 'maintainers_count': 0, 'dependencies_count': 0,
                    'has_install_script': 0, 'downloads': 0, 'description': '',
                    'has_repository': 0, 'keywords_count': 0, 'readme_length': 0,
                    'days_since_created': 0}
        return None
    except:
        return None

print('fetch_npm_full 함수 정의 완료')

## 2. Hard Negative Sampling — 진짜 어려운 정상 패키지 수집

In [ ]:
# ── 기존 benign CSV 로드 ──────────────────────────────────────
existing_benign = pd.read_csv('data/processed/benign_packages.csv')
existing_names  = set(existing_benign['name'].tolist())
print(f'기존 benign: {len(existing_benign)}건')

# ── Hard Negative 이름 목록 생성 ──────────────────────────────
# 전략 1: 인기 패키지의 합법적 파생 패키지 (유사 이름이지만 정상)
POPULAR_CORE = ['react','express','lodash','axios','webpack',
                'babel','eslint','typescript','vue','angular',
                'jquery','moment','chalk','commander','dotenv']

SUFFIXES = ['-utils','-helper','-core','-extra','-plus',
            '-lite','-mini','-native','-next','-cli',
            '-tools','-kit','-ui','-hooks','-plugin']
PREFIXES = ['node-','mini-','micro-','fast-','tiny-',
            'super-','ultra-','pro-','slim-','lean-']

hard_negative_candidates = set()
for pkg in POPULAR_CORE:
    for suf in SUFFIXES:
        hard_negative_candidates.add(f'{pkg}{suf}')
    for pre in PREFIXES:
        hard_negative_candidates.add(f'{pre}{pkg}')

# 전략 2: 스코프 패키지 (거의 항상 정상)
SCOPED = [
    '@babel/core','@babel/cli','@babel/preset-env','@babel/preset-react',
    '@babel/preset-typescript','@babel/parser','@babel/traverse',
    '@vue/cli','@vue/cli-service','@vue/compiler-sfc','@vue/reactivity',
    '@types/node','@types/react','@types/express','@types/lodash',
    '@types/jest','@types/mocha','@types/chai',
    '@angular/core','@angular/cli','@angular/common',
    '@nestjs/core','@nestjs/common','@nestjs/platform-express',
    '@aws-sdk/client-s3','@aws-sdk/client-dynamodb',
    '@google-cloud/storage','@google-cloud/bigquery',
    '@sveltejs/kit','@sveltejs/vite-plugin-svelte',
    '@tanstack/react-query','@tanstack/vue-query',
    '@emotion/react','@emotion/styled',
    '@mui/material','@mui/icons-material',
]
hard_negative_candidates.update(SCOPED)

# 전략 3: 설치 스크립트 있는 합법 패키지 (모델이 가장 헷갈려하는 케이스)
LEGIT_WITH_INSTALL = [
    'node-gyp','node-pre-gyp','prebuild-install','node-addon-api',
    'sharp','canvas','bcrypt','argon2','sodium-native',
    'sqlite3','better-sqlite3','leveldown','rocksdb',
    'fsevents','iohook','robotjs','node-hid',
    'serialport','node-usb','bluetooth-serial-port',
]
hard_negative_candidates.update(LEGIT_WITH_INSTALL)

new_targets = [n for n in hard_negative_candidates if n not in existing_names]
print(f'Hard Negative 수집 대상: {len(new_targets)}건')

# ── npm에서 실제 존재하는 것만 수집 ─────────────────────────
hard_negatives = []
for i, name in enumerate(new_targets):
    meta = fetch_npm_full(name, label=0)
    # 존재하고 다운로드가 어느 정도 있는 것만 benign으로 인정
    if meta and meta.get('exists') and meta.get('downloads', 0) >= 50:
        hard_negatives.append(meta)
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(new_targets)} 확인 | 수집: {len(hard_negatives)}건', end='\r')
    time.sleep(0.08)

print(f'\nHard Negative 수집 완료: {len(hard_negatives)}건')
print(f'  - 스코프 패키지: {sum(1 for r in hard_negatives if r["name"].startswith("@"))}건')
print(f'  - 설치스크립트 있는 정상: {sum(1 for r in hard_negatives if r["has_install_script"]==1)}건')

## 3. 전체 데이터셋 구성

In [ ]:
# ── 악성 데이터 로드 ──────────────────────────────────────────
MAL_FILES = {
    'maloss':        'data/processed/maloss_npm_malicious.csv',
    'backstabbers':  'data/processed/backstabbers_npm.csv',
    'advisory':      'data/processed/npm_advisory.csv',
    'hallucination': 'data/processed/llm_hallucinated_packages.csv',
}
mal_frames = []
for key, path in MAL_FILES.items():
    if not os.path.exists(path): continue
    df = pd.read_csv(path); df['label'] = 1
    mal_frames.append(df)
    print(f'[OK] {key}: {len(df)}건')

mal_all = pd.concat(mal_frames, ignore_index=True).drop_duplicates(subset=['name'])

# ── benign 합치기 (기존 + Hard Negative) ─────────────────────
hard_neg_df   = pd.DataFrame(hard_negatives)
benign_all    = pd.concat([existing_benign, hard_neg_df], ignore_index=True)
benign_all    = benign_all.drop_duplicates(subset=['name'])
benign_all['label'] = 0

print(f'\nBenign 합계: {len(benign_all)}건')
print(f'  기존: {len(existing_benign)}건 | Hard Negative: {len(hard_negatives)}건')

# ── 샘플링 (악성 최대 5000건, 정상 전체 사용) ────────────────
n_mal = min(5000, len(mal_all))
n_ben = len(benign_all)
mal_sampled = mal_all.sample(n_mal, random_state=RANDOM_SEED)
df = pd.concat([mal_sampled, benign_all]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# 컨텍스트 컬럼이 없는 행은 기본값으로 채우기
for col in ['description','has_repository','keywords_count','readme_length','days_since_created']:
    if col not in df.columns:
        df[col] = '' if col == 'description' else 0

print(f'\n최종 데이터셋: 악성 {n_mal}건 / 정상 {n_ben}건 = {n_mal/n_ben:.1f}:1')
print(f'합계: {len(df)}건')

## 4. 메타데이터 피처 추출 (12차원)

In [ ]:
from difflib import SequenceMatcher

POPULAR = ['react','express','lodash','axios','webpack','babel','eslint',
           'typescript','vue','angular','jquery','moment','chalk','commander',
           'dotenv','fastify','koa','next','vite','prisma']

def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if a[i-1]==b[j-1] else 1+min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n]

def min_edit_dist(name):
    core = str(name).lower().lstrip('@').split('/')[0]
    return min(levenshtein(core, p) for p in POPULAR)

def name_sim(name):
    return max(SequenceMatcher(None, str(name).lower(), p).ratio() for p in POPULAR)

def suspicious(name):
    patterns = [r'\d{3,}$', r'_{2,}', r'-{2,}', r'^[a-z]{1,2}$']
    return int(any(re.search(p, str(name)) for p in patterns))

def safe_num(val, default=0):
    try:
        v = float(val)
        return default if np.isnan(v) else v
    except:
        return float(default)

def extract_meta(row):
    name   = str(row.get('name', ''))
    d      = safe_num(row.get('downloads', 0))
    edit_d = min_edit_dist(name)
    return [
        np.log1p(d),
        name_sim(name),
        suspicious(name),
        int(not str(row.get('author', '')).strip()),
        int(safe_num(row.get('has_install_script', 0))),
        int(safe_num(row.get('dependencies_count', 0)) > 20),
        min(safe_num(row.get('maintainers_count', 0)), 20),
        min(safe_num(row.get('dependencies_count', 0)), 50),
        int(not bool(row.get('exists', True))),
        min(edit_d, 10),       # min_edit_dist  ← NEW
        int(edit_d == 0),      # is_exact_popular ← NEW
        int(name.startswith('@')),  # is_scoped ← NEW
    ]

meta_X = np.array([extract_meta(row) for _, row in df.iterrows()], dtype=np.float32)
print(f'메타 피처: {meta_X.shape}  (12차원)')

# min_edit_dist 분포 확인
edit_dists = meta_X[:, 9]
print(f'  edit_dist=0 (유명패키지 자체):  {(edit_dists==0).sum()}건')
print(f'  edit_dist=1 (오타 1개):         {(edit_dists==1).sum()}건')
print(f'  edit_dist=2 (오타 2개):         {(edit_dists==2).sum()}건')
print(f'  edit_dist>=3 (완전 다른 이름):  {(edit_dists>=3).sum()}건')

## 5. 컨텍스트 피처 수집 (description 없는 악성 패키지 확인)

악성 패키지의 description 비어있는 비율이 핵심 신호입니다.

In [ ]:
# description이 CSV에 없는 경우 npm에서 보완 수집
if 'description' not in df.columns or df['description'].isna().all():
    print('description 수집 중...')
    descs = []
    for i, name in enumerate(df['name'].tolist()):
        try:
            req = urllib.request.Request(
                f'https://registry.npmjs.org/{name}',
                headers={'User-Agent': 'Mozilla/5.0'}
            )
            with urllib.request.urlopen(req, timeout=4) as r:
                descs.append(json.loads(r.read()).get('description', ''))
        except:
            descs.append('')
        if (i+1) % 200 == 0:
            print(f'  {i+1}/{len(df)} 완료', end='\r')
        time.sleep(0.05)
    df['description'] = descs

df['description'] = df['description'].fillna('')

# 클래스별 description 분포 분석
print('\n=== description 분포 (핵심 신호 확인) ===')
for label, lname in [(1,'악성'), (0,'정상')]:
    sub   = df[df['label']==label]
    empty = (sub['description'].str.strip() == '').sum()
    print(f'  {lname}: {empty}/{len(sub)}건 비어있음 ({empty/len(sub)*100:.1f}%)')

# CodeBERT 입력 텍스트 구성 (description + 컨텍스트)
def build_text(row):
    parts = []
    desc = str(row.get('description', '')).strip()
    name = str(row.get('name', ''))
    if desc:
        parts.append(desc)
    if row.get('keywords_count', 0) and row.get('keywords_count', 0) > 0:
        parts.append(f'keywords: {int(row["keywords_count"])} tags')
    if row.get('has_repository', 0):
        parts.append('repository: available')
    if not parts:  # 아무 정보도 없으면 패키지명
        parts.append(f'package {name}')
    return ' | '.join(parts)

texts = [build_text(row) for _, row in df.iterrows()]
with_desc = sum(1 for _, row in df.iterrows() if str(row.get('description','')).strip())
print(f'\nCodeBERT 입력 구성:')
print(f'  description 사용: {with_desc}건')
print(f'  패키지명 fallback: {len(texts)-with_desc}건')

## 6. CodeBERT 임베딩 (768차원, GPU 기준 10~20분)

In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'microsoft/codebert-base'
BATCH_SIZE = 64

print(f'CodeBERT 로드 중... (device: {DEVICE})')
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()
print('로드 완료')

def embed_batch(text_list, batch_size=BATCH_SIZE):
    all_emb = []
    total, t0 = len(text_list), time.time()
    for i in range(0, total, batch_size):
        batch  = text_list[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', max_length=512,
                           truncation=True, padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = bert_model(**inputs)
        all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())
        done    = min(i+batch_size, total)
        elapsed = time.time() - t0
        eta     = elapsed / done * (total - done) if done > 0 else 0
        print(f'  {done}/{total}  경과 {elapsed:.0f}s  남은 예상 {eta:.0f}s', end='\r')
    print()
    return np.vstack(all_emb)

print(f'\n임베딩 시작: {len(texts)}건')
emb_X = embed_batch(texts)
print(f'완료: {emb_X.shape}')

## 7. 피처 결합 & 5-Fold CV

In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error, classification_report

# 결합: 768(CodeBERT) + 12(meta) = 780차원
X = np.hstack([emb_X, meta_X])
y = df['label'].values
print(f'피처 행렬: {X.shape}')
print(f'클래스: 악성 {y.sum()}건 / 정상 {(y==0).sum()}건')

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_results = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    sc      = StandardScaler()
    X_tr_s  = sc.fit_transform(X_tr)
    X_val_s = sc.transform(X_val)

    neg, pos = (y_tr==0).sum(), (y_tr==1).sum()
    m = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=neg/pos, subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
        eval_metric='logloss', random_state=RANDOM_SEED, verbosity=0,
    )
    m.fit(X_tr_s, y_tr, eval_set=[(X_val_s, y_val)], verbose=False)

    y_pred  = m.predict(X_val_s)
    y_proba = m.predict_proba(X_val_s)[:, 1]
    fold_results.append({
        'f1':  f1_score(y_val, y_pred),
        'auc': roc_auc_score(y_val, y_proba),
        'mae': mean_absolute_error(y_val, y_proba),
        'n_ben': (y_val==0).sum(),
    })
    r = fold_results[-1]
    print(f'Fold {fold+1}: F1={r["f1"]:.4f} AUC={r["auc"]:.4f} benign_test={r["n_ben"]}건')

print('\n' + '='*55)
f1s  = [r['f1']  for r in fold_results]
aucs = [r['auc'] for r in fold_results]
maes = [r['mae'] for r in fold_results]
print(f'  5-Fold F1  : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'  5-Fold AUC : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'  5-Fold MAE : {np.mean(maes):.4f} ± {np.std(maes):.4f}')
print('='*55)

# 최종 모델: 전체 재학습
print('\n최종 모델 전체 데이터 재학습 중...')
scaler = StandardScaler()
X_all_s = scaler.fit_transform(X)
neg, pos = (y==0).sum(), (y==1).sum()
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=neg/pos, subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
    eval_metric='logloss', random_state=RANDOM_SEED, verbosity=0,
)
model.fit(X_all_s, y)
f1  = float(np.mean(f1s))
auc = float(np.mean(aucs))
mae = float(np.mean(maes))
print('재학습 완료')

## 8. 저장 & 다운로드

In [ ]:
import pickle

os.makedirs('models', exist_ok=True)
PKL_PATH = 'models/ensemble_xgb_v2.pkl'

bundle = {
    'model':      model,
    'scaler':     scaler,
    'metrics':    {'f1': f1, 'auc': auc, 'mae': mae},
    'model_type': 'ensemble_codebert_xgboost_v2',
    'feat_dim':   X.shape[1],
    'n_benign':   int((y==0).sum()),
    'n_malicious':int((y==1).sum()),
    'hard_negative_count': len(hard_negatives),
}

with open(PKL_PATH, 'wb') as f:
    pickle.dump(bundle, f)

print(f'저장 완료: {PKL_PATH}')
print(f'파일 크기: {os.path.getsize(PKL_PATH)/1024/1024:.1f} MB')
print(f'피처 차원: {X.shape[1]} (CodeBERT 768 + meta 12)')
print(f'Hard Negative 포함 benign: {bundle["n_benign"]}건')
print(f'5-Fold CV F1: {f1:.4f} ± {np.std(f1s):.4f}')

In [ ]:
from google.colab import files
files.download(PKL_PATH)
print('다운로드 완료 — models/ensemble_xgb_v2.pkl 로 저장하세요')